In [1]:
import json
import os
import random
import re

from importlib.util import find_spec
from pathlib import Path

import numpy as np
import torch
from qwen_vl_utils import process_vision_info
from torchvision.transforms.functional import to_pil_image
from transformers import (
    AutoProcessor,
    GenerationConfig,
    Qwen2VLForConditionalGeneration,
)

In [2]:
# ---------------------------- Experiment settings ----------------------------
SEED = 42
VERBOSE = 2

# Path
MODEL_PATH = "JZPeterPan/MedVLM-R1"
HF_CACHE = "/data/huggingface_cache"
os.environ["HF_HOME"] = HF_CACHE

In [3]:
def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root starting from '{current}'")

PROJECT_ROOT = find_project_root()
SAMPLE_ROOT = PROJECT_ROOT / "data" / "OmniMedVQA" / "sample_mri"
QUESTION_PATH = SAMPLE_ROOT / "question.json"

OUTPUT_DIRECTORY = PROJECT_ROOT / "result" / "MedVLM-R1" / "bias_field_attack" / "cps_8_eps_0p3" / "batch_0"
ADVERSARIAL_IMAGE_DIRECTORY  = OUTPUT_DIRECTORY / "attacked_images"

HISTORY_FIELDS = [
    "question_id",
    "step",
    "loss",
    "prob_A",
    "prob_B",
    "prob_C",
    "prob_D",
]

def load_completed_ids(result_path):
    if not result_path.exists():
        return set()

    with result_path.open(encoding="utf-8") as file:
        return {
            str(json.loads(line)["question_id"])
            for line in file
            if line.strip()
        }
        
with QUESTION_PATH.open(encoding="utf-8") as file:
    all_samples = json.load(file)

all_mri_samples = [sample for sample in all_samples if sample.get("modality") == "MRI"]

In [4]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Submit this script to a GPU node."
    )

device = "cuda"
dtype = torch.bfloat16
computation_dtype = torch.float32
flash_attn_available = find_spec("flash_attn") is not None
attn_implementation = "flash_attention_2" if flash_attn_available else "sdpa"
print(f"Using attention implementation: {attn_implementation}")
print(f"Using device: {device}")
print(f"Using dtype: {dtype}")

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    dtype=dtype,
    attn_implementation=attn_implementation,
    device_map="auto",
)
model.eval()
model.requires_grad_(False)
model.config.use_cache = False

processor = AutoProcessor.from_pretrained(MODEL_PATH)
image_processor = processor.image_processor
tokenizer = processor.tokenizer

generation_config = GenerationConfig(
    max_new_tokens=1024,
    do_sample=False,
    num_return_sequences=1,
    pad_token_id=151643,
)

print("Model and processor loaded successfully.")

Using attention implementation: sdpa
Using device: cuda
Using dtype: torch.bfloat16


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Model and processor loaded successfully.


In [5]:
QUESTION_TEMPLATE = """
    {Question}
    Your task:
    1. Think through the question step by step, enclose your reasoning process in <think>...</think> tags.
    2. Then provide the correct single-letter choice (A, B, C, D,...) inside <answer>...</answer> tags.
    3. No extra information or text outside of these tags.
    """

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def extract_answer(output_text, tag="answer"):
    match = re.search(rf"<{tag}>\s*(.*?)\s*</{tag}>", output_text, re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else None

def tensor_to_pil(image_tensor):
    if image_tensor.ndim == 4:
        if image_tensor.shape[0] != 1:
            raise ValueError("Only batch size one is supported")
        image_tensor = image_tensor.squeeze(0)
    if image_tensor.ndim != 3:
        raise ValueError(
            f"Expected image shape [C, H, W] or [1, C, H, W], but got {tuple(image_tensor.shape)}.")
    return to_pil_image(image_tensor.detach().cpu().float().clamp(0, 1))

In [6]:
def build_message(question, image_source):
    
    if isinstance(image_source, (str, Path)):
        image_source = Path(image_source).resolve().as_uri()
    elif isinstance(image_source, torch.Tensor):
        image_source = tensor_to_pil(image_source)
        
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_source},
            {"type": "text", "text": QUESTION_TEMPLATE.format(Question=question)},
        ],
    }]

@torch.inference_mode()
def run_model(question, image):

    message = build_message(question, image)
    text = processor.apply_chat_template(message, tokenize=False, add_generation_prompt=True) 
    image_inputs, video_inputs = process_vision_info(message)
    
    inputs_adv = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(device)
    
    generated_ids = model.generate(**inputs_adv, use_cache=True, generation_config=generation_config)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs_adv.input_ids, generated_ids)]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    
    return output_text[0]

In [ ]:
saved_image_results = []

for sample in all_mri_samples:
    sample_id = sample["id"]
    question = sample["problem"]
    correct_answer = sample["solution"]
    safe_id = sample_id.replace(":", "_")

    image_path = ADVERSARIAL_IMAGE_DIRECTORY / f"{safe_id}_biased.png"

    if not image_path.exists():
        print(f"Skipped {sample_id}, image not found at {image_path}.")
        continue

    model_output = run_model(question=question, image=image_path)

    predicted_answer = extract_answer(model_output)
    attack_success = predicted_answer != correct_answer
    
    results = {
        "id": sample_id,
        "question": question,
        "image_path": str(image_path),
        "correct_answer": correct_answer,
        "predicted_answer": predicted_answer,
        "attack_success": attack_success,
        "model_output": repr(model_output),
    }

    saved_image_results.append(results)

    for key, value in results.items():
        print(f"{key}: {value}")

# saved_image_results

In [12]:
import pandas as pd
saved_image_results_df = pd.DataFrame(saved_image_results)
failed = saved_image_results_df[saved_image_results_df["attack_success"] == False]
print(len(failed), "samples failed to attack out of", len(saved_image_results_df))
failed

7 samples failed to attack out of 50


,id,question,image_path,correct_answer,predicted_answer,attack_success,model_output
8,mod-mri:000008,What type of imaging was employed to take this...,/home/euro/code/sheffield/dissertation/attack-...,D,D,False,"""<think>\n The image appears to be a radiog..."
25,mod-mri:000025,What type of imaging technique was employed fo...,/home/euro/code/sheffield/dissertation/attack-...,D,D,False,'<think>\n The image appears to be a medica...
29,mod-mri:000029,What modality of imaging was utilized to acqui...,/home/euro/code/sheffield/dissertation/attack-...,C,C,False,'<think>\n The image appears to be a mammog...
33,mod-mri:000033,What type of imaging scan was utilized to obta...,/home/euro/code/sheffield/dissertation/attack-...,D,D,False,'<think>\n The image appears to be a radiog...
39,mod-mri:000039,What imaging modality was used to capture this...,/home/euro/code/sheffield/dissertation/attack-...,B,B,False,'<think>\n The image appears to be a medica...
44,mod-mri:000044,What type of imaging device was utilized to ac...,/home/euro/code/sheffield/dissertation/attack-...,B,B,False,'<think>\n The question asks to identify th...
48,mod-mri:000048,What imaging modality was used to capture this...,/home/euro/code/sheffield/dissertation/attack-...,C,C,False,'<think>\n The image appears to be a medica...
